In [1]:
import jax 
import jax.numpy as jnp
import numpy as np
np.set_printoptions(threshold=3000)
np.set_printoptions(linewidth=300)
np.set_printoptions(precision=2)

import os 
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.25"

print(jax.devices())

[CudaDevice(id=0)]


In [2]:
# FIFO buffer not parallelisable, correct?
def replicated_delay_mechanism(state, x):
    """
    :param state: 
    :param x: 
    :return: 
    """
    delay_buffer, out, delays = state
    delay_buffer = jnp.concatenate(
        [jax.lax.expand_dims(x, (0,)), delay_buffer[:-1,:]],
        0
    )
    n_inputs = delay_buffer.shape[1] 
    n_delays = delays.shape[-1] // n_inputs
    print(f'{n_inputs=}', f'{n_delays=}')
    col_idx = jnp.arange(n_inputs).repeat(n_delays)
    delayed_x = delay_buffer[delays, col_idx]
    out = delayed_x
    return (delay_buffer, out, delays), (out, delay_buffer)

In [3]:
def delay_mich_fft_krep(input, K):
    '''
    :param input: 1D array of shape (sim_len,)
    :param K: 2D array of shape (n_rep, d_max). Each row is a one-hot encoded delay
    :return: 2D array of shape (n_rep, sim_len+n_rep)
    '''
    print(f'{input.shape=}')
    print(f'{K.shape=}')
    d_max = K.shape[-1]
    sim_len = input.shape[-1]
    input_fft = jnp.fft.rfft(jnp.pad(input, (0, d_max)))
    K_fft = jnp.fft.rfft(jnp.pad(K, ((0,0), (0, sim_len))))
    I_fft = K_fft * input_fft
    I = jnp.fft.irfft(I_fft)[:,:input.shape[0]]
    return I

v_delay_mich_fft_krep = jax.vmap(delay_mich_fft_krep, in_axes=(0, 0))
v_v_delay_mich_fft_krep = jax.vmap(v_delay_mich_fft_krep, in_axes=(None, 0))

In [4]:
def delay_mich_fft_k(input, K):
    '''
    :param input: 1D array of shape (sim_len,)
    :param K: 1D array of shape (d_max,). n_rep-hot encoded delays
    :return: 1D array of shape (sim_len+d_max,)
    '''
    print(f'{input.shape=}')
    print(f'{K.shape=}')
    d_max = K.shape[-1]
    sim_len = input.shape[-1]
    input_fft = jnp.fft.rfft(jnp.pad(input, (0, d_max)))
    K_fft = jnp.fft.rfft(jnp.pad(K, (0, sim_len)))
    I_fft = K_fft * input_fft
    I = jnp.fft.irfft(I_fft)[:input.shape[0]]
    return I

v_delay_mich_fft_k = jax.vmap(delay_mich_fft_k, in_axes=(0, 0))
v_v_delay_mich_fft_k = jax.vmap(v_delay_mich_fft_k, in_axes=(None, 0))

In [5]:
def uniform_delay_generation(key, n_out, n_in, n_rep, d_max):
    key, subkey = jax.random.split(key)
    discrete_delays = (
        jax.random.randint(subkey, (n_out, n_in*n_rep),
                           minval=0,
                           maxval=d_max + 1))
    return key, discrete_delays


In [6]:
def uniform_delay_k_generation(key, n_out, n_in, n_rep, d_max):
    key, subkey = jax.random.split(key)
    discrete_delays = (
        jax.random.randint(subkey, (n_out, n_in, n_rep),
                           minval=0,
                           maxval=d_max + 1))
    return key, discrete_delays

In [7]:
def delay_k_generation(n_in, n_rep, n_out, d_max):
    delays = jnp.zeros((n_in, n_rep), dtype='int32')
    offset = 0
    for i in range(n_in):
        d = jnp.arange(offset, n_rep+offset)
        delays = delays.at[i].set(d)
        if n_rep + offset == d_max:
            offset = 0
        else:
            offset += 1
    print(delays.shape)
    print(delays)
    for i in range(n_in):
        assert jnp.unique(delays[i]).shape[0] == n_rep, f'{i=} {jnp.unique(delays[i])}'
    # WHY DO WE DO THIS TYPE OF INITIALISATION? IS IT JUST FOR TESTING?
    def mo_delays(delays, n_out):
        delays_mo = [(delays + i) % d_max for i in range(n_out)]
        return jnp.stack(delays_mo)

    return mo_delays(delays, n_out)

def delay_from_kernel_to_denram(delays):
    return delays.reshape(n_out, n_in*n_rep)

In [8]:
n_in=5
n_rep=2 # this gives us only one delay per input
n_out=3
d_max=5

dk=delay_k_generation(n_in, n_rep, n_out, d_max)
print(dk.shape)
dk = delay_from_kernel_to_denram(dk)
print(dk)

key,dk_alt=uniform_delay_k_generation(jax.random.PRNGKey(0), n_out, n_in, n_rep, d_max)
print(dk_alt.shape)
# key,dk_alt=uniform_delay_generation(jax.random.PRNGKey(0), n_out, n_in, n_rep, d_max)
dk_alt = delay_from_kernel_to_denram(dk_alt)
print(dk_alt)

(5, 2)
[[0 1]
 [1 2]
 [2 3]
 [3 4]
 [0 1]]
(3, 5, 2)
[[0 1 1 2 2 3 3 4 0 1]
 [1 2 2 3 3 4 4 0 1 2]
 [2 3 3 4 4 0 0 1 2 3]]
(3, 5, 2)
[[2 4 4 3 5 3 4 4 5 0]
 [4 4 1 1 5 5 1 3 4 3]
 [2 5 0 1 4 1 3 2 5 4]]


In [9]:
def k_generation(n_in, n_rep, n_out, d_max, delays):
    out_idx = jnp.tile(jnp.ones((n_in, n_rep), dtype='int32'), (n_out, 1, 1)) * jnp.arange(n_out)[:, None, None]
    in_idx = jnp.ones((n_in, n_rep), dtype='int32') * jnp.arange(n_in)[:, None]
    rep_idx = jnp.tile(jnp.arange(n_rep), (n_in, 1))
    K_rep = jnp.zeros((n_out, n_in, n_rep, d_max), dtype='int32')
    K_rep = K_rep.at[out_idx, in_idx, rep_idx, delays].set(1)
    K = jnp.zeros((n_out, n_in, d_max), dtype='int32')
    K = K.at[out_idx, in_idx, delays].set(1)
    print(jnp.array_equal(K_rep.sum(2), K))
    return K, K_rep




In [10]:
n_in = 3
n_out = 2
n_rep = 4
sim_len = 15
d_max = 5

mc_input = jnp.arange(1, sim_len+1)[:, None] + jnp.arange(n_in)[None, :] * sim_len
print(mc_input.T, "= multi_channel_input.T")
mc_input_k = mc_input.T

dk = delay_k_generation(n_in, n_rep, n_out, d_max)
print(dk.shape, "= dk.shape")
d_dr = delay_from_kernel_to_denram(dk)
print(d_dr.shape, "= d_dr.shape")

K, K_rep = k_generation(n_in, n_rep, n_out, d_max, dk)

[[ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15]
 [16 17 18 19 20 21 22 23 24 25 26 27 28 29 30]
 [31 32 33 34 35 36 37 38 39 40 41 42 43 44 45]] = multi_channel_input.T
(3, 4)
[[0 1 2 3]
 [1 2 3 4]
 [0 1 2 3]]
(2, 3, 4) = dk.shape
(2, 12) = d_dr.shape
True


In [11]:
out_dr = jnp.zeros((n_out, n_in*n_rep), dtype='int32')
delay_buffer = jnp.zeros((d_max, n_in), dtype='int32')
state = (delay_buffer, out_dr, d_dr)
_, (out_dr, delay_buffer) = jax.lax.scan(replicated_delay_mechanism, state, mc_input)
print(out_dr.shape)

n_inputs=3 n_delays=4
(15, 2, 12)


In [12]:
out_krep = v_v_delay_mich_fft_krep(mc_input_k, K_rep)
print(out_krep.shape)
out_krep_sum = out_krep.round(2).astype('int32').sum(2)
print(out_krep_sum.shape)

input.shape=(15,)
K.shape=(4, 5)
(2, 3, 4, 15)
(2, 3, 15)


In [13]:
out_k = v_v_delay_mich_fft_k(mc_input_k, K)
out_k = out_k.round(2).astype('int32')
print(out_k.shape)

input.shape=(15,)
K.shape=(5,)
(2, 3, 15)


In [14]:
def out_from_denram_to_k(out):
    out_reshaped = jnp.swapaxes(out, 0, 1).reshape(n_out, sim_len, n_in, n_rep)
    out_reshaped = jnp.swapaxes(out_reshaped, 1, 2)
    return out_reshaped.sum(axis=-1)

out_k_from_dr = out_from_denram_to_k(out_dr)
print(out_k_from_dr.shape)

(2, 3, 15)


In [15]:
print(jnp.array_equal(out_k_from_dr, out_k))
print(jnp.array_equal(out_k_from_dr, out_krep_sum))
print(jnp.array_equal(out_k, out_krep_sum))

True
True
True


In [16]:
print(out_k_from_dr)

[[[  1   3   6  10  14  18  22  26  30  34  38  42  46  50  54]
  [  0  16  33  51  70  74  78  82  86  90  94  98 102 106 110]
  [ 31  63  96 130 134 138 142 146 150 154 158 162 166 170 174]]

 [[  0   1   3   6  10  14  18  22  26  30  34  38  42  46  50]
  [ 16  17  34  52  71  75  79  83  87  91  95  99 103 107 111]
  [  0  31  63  96 130 134 138 142 146 150 154 158 162 166 170]]]


In [17]:
print(out_k)

[[[  1   3   6  10  14  18  22  26  30  34  38  42  46  50  54]
  [  0  16  33  51  70  74  78  82  86  90  94  98 102 106 110]
  [ 31  63  96 130 134 138 142 146 150 154 158 162 166 170 174]]

 [[  0   1   3   6  10  14  18  22  26  30  34  38  42  46  50]
  [ 16  17  34  52  71  75  79  83  87  91  95  99 103 107 111]
  [  0  31  63  96 130 134 138 142 146 150 154 158 162 166 170]]]


In [16]:
n_in = 700
n_out = 20
n_rep = 16
sim_len = 150
d_max = 100


# mc_input = jnp.arange(1, sim_len+1)[:, None] + jnp.arange(n_in)[None, :] * sim_len
# print(mc_input.T, "= multi_channel_input.T")
# mc_input_k = mc_input.T
key = jax.random.PRNGKey(0)
mc_input = jax.random.randint(key, minval=0, maxval=100, shape=(sim_len, n_in))
mc_input_k = mc_input.T
print(mc_input_k.shape, "= mc_input_k.shape")

dk = delay_k_generation(n_in, n_rep, n_out, d_max)
print(dk.shape, "= dk.shape")
d_dr = delay_from_kernel_to_denram(dk)
print(d_dr.shape, "= d_dr.shape")

K, K_rep = k_generation(n_in, n_rep, n_out, d_max, dk)

(700, 150) = mc_input_k.shape
(700, 16)
[[ 0  1  2 ... 13 14 15]
 [ 1  2  3 ... 14 15 16]
 [ 2  3  4 ... 15 16 17]
 ...
 [17 18 19 ... 30 31 32]
 [18 19 20 ... 31 32 33]
 [19 20 21 ... 32 33 34]]
(20, 700, 16) = dk.shape
(20, 11200) = d_dr.shape
True


In [17]:
v_delay_mich_fft_k = jax.vmap(delay_mich_fft_k, in_axes=(0, 0))
jv_delay_mich_fft_k = jax.jit(v_delay_mich_fft_k)
vjv_delay_mich_fft_k = jax.vmap(jv_delay_mich_fft_k, in_axes=(None, 0))
jvjv_delay_mich_fft_k = jax.jit(vjv_delay_mich_fft_k)

In [18]:
j_delay_mich_fft_k = jax.jit(delay_mich_fft_k)
vj_delay_mich_fft_k = jax.vmap(j_delay_mich_fft_k, in_axes=(0, 0))
jvj_delay_mich_fft_k = jax.jit(vj_delay_mich_fft_k)
vjvj_delay_mich_fft_k = jax.vmap(jvj_delay_mich_fft_k, in_axes=(None, 0))
jvjvj_delay_mich_fft_k = jax.jit(vjvj_delay_mich_fft_k)

In [19]:
v_delay_mich_fft_krep = jax.vmap(delay_mich_fft_krep, in_axes=(0, 0))
jv_delay_mich_fft_krep = jax.jit(v_delay_mich_fft_krep)
vjv_delay_mich_fft_krep = jax.vmap(jv_delay_mich_fft_krep, in_axes=(None, 0))
jvjv_delay_mich_fft_krep = jax.jit(vjv_delay_mich_fft_krep)

In [20]:
j_delay_mich_fft_krep = jax.jit(delay_mich_fft_krep)
vj_delay_mich_fft_krep = jax.vmap(j_delay_mich_fft_krep, in_axes=(0, 0))
jvj_delay_mich_fft_krep = jax.jit(vj_delay_mich_fft_krep)
vjvj_delay_mich_fft_krep = jax.vmap(jvj_delay_mich_fft_krep, in_axes=(None, 0))
jvjvj_delay_mich_fft_krep = jax.jit(vjvj_delay_mich_fft_krep)

In [21]:
def delay_krep_jvjv(mc_input_k, K_rep):
    return jvjv_delay_mich_fft_krep(mc_input_k, K_rep).round(2).astype('int32').sum(2)

def delay_krep_vjvj(mc_input_k, K_rep):
    return vjvj_delay_mich_fft_krep(mc_input_k, K_rep).round(2).astype('int32').sum(2)

def delay_krep_jvjvj(mc_input_k, K_rep):
    return jvjvj_delay_mich_fft_krep(mc_input_k, K_rep).round(2).astype('int32').sum(2)


j_delay_krep_jvjv = jax.jit(delay_krep_jvjv)
j_delay_krep_vjvj = jax.jit(delay_krep_vjvj)
j_delay_krep_jvjvj = jax.jit(delay_krep_jvjvj)

In [22]:
j_out_from_denram_to_k = jax.jit(out_from_denram_to_k)

def wrapper_delay_denram(n_in, n_rep, n_out, d_max, mc_input):
    global d_dr
    out_dr = jnp.zeros((n_out, n_in*n_rep), dtype='int32')
    delay_buffer = jnp.zeros((d_max, n_in), dtype='int32')
    state = (delay_buffer, out_dr, d_dr)
    _, (out_dr, delay_buffer) = jax.lax.scan(replicated_delay_mechanism, state, mc_input)
    return j_out_from_denram_to_k(out_dr)

j_wrapper_delay_denram = jax.jit(wrapper_delay_denram, static_argnums=(0, 1, 2, 3))

In [25]:
type(d_dr)

jaxlib.xla_extension.ArrayImpl

In [26]:
type(jnp.array([1,2,3]))

jaxlib.xla_extension.ArrayImpl

In [23]:
# warm-up
out_dr = j_wrapper_delay_denram(n_in, n_rep, n_out, d_max, mc_input)
out_krep_jvjvj = j_delay_krep_jvjvj(mc_input_k, K_rep)
out_krep_jvjv = j_delay_krep_jvjv(mc_input_k, K_rep)
out_k_jvjvj = jvjvj_delay_mich_fft_k(mc_input_k, K)
out_k_jvjv = jvjv_delay_mich_fft_k(mc_input_k, K)


print(jnp.array_equal(out_dr, out_krep_jvjvj))
print(jnp.array_equal(out_dr, out_k_jvjvj.round(2).astype('int32')))
print(jnp.array_equal(out_dr, out_krep_jvjv))
print(jnp.array_equal(out_dr, out_k_jvjv.round(2).astype('int32')))

n_inputs=700 n_delays=16
input.shape=(150,)
K.shape=(16, 100)
input.shape=(150,)
K.shape=(16, 100)
input.shape=(150,)
K.shape=(100,)
input.shape=(150,)
K.shape=(100,)
True
True
True
True


In [28]:
%timeit j_wrapper_delay_denram(n_in, n_rep, n_out, d_max, mc_input)

1.38 ms ± 172 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [ ]:
%timeit j_delay_krep_jvjvj(mc_input_k, K_rep)

2.98 ms ± 493 ns per loop (mean ± std. dev. of 7 runs, 100 loops each)


: 

In [43]:
%timeit jvjvj_delay_mich_fft_k(mc_input_k, K)

83.8 μs ± 36.1 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [44]:
%timeit j_delay_krep_jvjvj(mc_input_k, K_rep)

2.98 ms ± 554 ns per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [45]:
%timeit jvjv_delay_mich_fft_k(mc_input_k, K)

83.9 μs ± 8.52 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [24]:
denram_time = 1410 # in us
krep_time = 2990 # in us
k_time = 84.2 # in us
print(f'K allows to go {denram_time/k_time:.2f} times faster')
print(f'K_rep is {krep_time/k_time:.2f} times slower than K')

K allows to go 16.75 times faster
K_rep is 35.51 times slower than K


In [25]:
def weight_generation(key, n_out, n_in, n_rep):
    w = jax.random.normal(key, (n_out, n_in, n_rep))
    return w

weight_from_kernel_to_denram = lambda w: w.reshape(n_out, n_in*n_rep)

In [26]:
def kw_generation(n_in, n_rep, n_out, d_max, delays, weights):
    out_idx = jnp.tile(jnp.ones((n_in, n_rep), dtype='int32'), (n_out, 1, 1)) * jnp.arange(n_out)[:, None, None]
    in_idx = jnp.ones((n_in, n_rep), dtype='int32') * jnp.arange(n_in)[:, None]
    rep_idx = jnp.tile(jnp.arange(n_rep), (n_in, 1))
    K_rep = jnp.zeros((n_out, n_in, n_rep, d_max), dtype='float32')
    K_rep = K_rep.at[out_idx, in_idx, rep_idx, delays].set(weights)
    K = jnp.zeros((n_out, n_in, d_max), dtype='float32')
    K = K.at[out_idx, in_idx, delays].set(weights)
    print(jnp.array_equal(K_rep.sum(2), K))
    return K, K_rep

In [ ]:
## HOW TO GENERATE THE KERNEL (w/ DELAYS AND WEIGHTS) FOR THE DENRAM!!!


n_in = 700
n_out = 20 # but what is n_out? Number of output neurons? Number of output channels?
n_rep = 16 # number of delays
sim_len = 150
d_max = 100


# mc_input = jnp.arange(1, sim_len+1)[:, None] + jnp.arange(n_in)[None, :] * sim_len
# print(mc_input.T, "= multi_channel_input.T")
# mc_input_k = mc_input.T
key = jax.random.PRNGKey(0)
mc_input = jax.random.randint(key, minval=0, maxval=2, shape=(sim_len, n_in))
print(mc_input)
mc_input_k = mc_input.T
print(mc_input_k.shape, "= mc_input_k.shape") #(n_in, sim_len)
print(mc_input.min(), mc_input.max(), "= min, max")

dk = delay_k_generation(n_in, n_rep, n_out, d_max)
print("dk", dk)
print(dk.shape, "= dk.shape")
d_dr = delay_from_kernel_to_denram(dk)
print(d_dr.shape, "= d_dr.shape")

w = weight_generation(key, n_out, n_in, n_rep)
w_dr = weight_from_kernel_to_denram(w)
print(w.shape, "= w.shape")
print(w_dr.shape, "= w_dr.shape")

K, K_rep = kw_generation(n_in, n_rep, n_out, d_max, dk, w)
print(K.shape, K_rep.shape) #Kw (n_out, n_in, d_max) K_rep (n_out, n_in, d_max)


print(jnp.array_equal(K, K_rep.sum(2)))

[[1 0 0 ... 0 0 0]
 [0 0 0 ... 1 1 1]
 [0 1 1 ... 1 1 0]
 ...
 [1 1 0 ... 0 0 0]
 [0 0 1 ... 0 0 1]
 [0 1 0 ... 0 1 0]]
(700, 150) = mc_input_k.shape
0 1 = min, max
(700, 16)
[[ 0  1  2 ... 13 14 15]
 [ 1  2  3 ... 14 15 16]
 [ 2  3  4 ... 15 16 17]
 ...
 [17 18 19 ... 30 31 32]
 [18 19 20 ... 31 32 33]
 [19 20 21 ... 32 33 34]]
dk [[[ 0  1  2 ... 13 14 15]
  [ 1  2  3 ... 14 15 16]
  [ 2  3  4 ... 15 16 17]
  ...
  [17 18 19 ... 30 31 32]
  [18 19 20 ... 31 32 33]
  [19 20 21 ... 32 33 34]]

 [[ 1  2  3 ... 14 15 16]
  [ 2  3  4 ... 15 16 17]
  [ 3  4  5 ... 16 17 18]
  ...
  [18 19 20 ... 31 32 33]
  [19 20 21 ... 32 33 34]
  [20 21 22 ... 33 34 35]]

 [[ 2  3  4 ... 15 16 17]
  [ 3  4  5 ... 16 17 18]
  [ 4  5  6 ... 17 18 19]
  ...
  [19 20 21 ... 32 33 34]
  [20 21 22 ... 33 34 35]
  [21 22 23 ... 34 35 36]]

 ...

 [[17 18 19 ... 30 31 32]
  [18 19 20 ... 31 32 33]
  [19 20 21 ... 32 33 34]
  ...
  [34 35 36 ... 47 48 49]
  [35 36 37 ... 48 49 50]
  [36 37 38 ... 49 50 51]]

 [[1

: 

In [28]:
print(K.shape, K_rep.shape)

(20, 700, 100) (20, 700, 16, 100)


In [29]:
def delay_denram_w(state, x):
    delay_buffer, out, delays, w = state
    delay_buffer = jnp.concatenate(
        [jax.lax.expand_dims(x, (0,)), delay_buffer[:-1,:]],
        0
    )
    n_inputs = delay_buffer.shape[1] 
    n_delays = delays.shape[-1] // n_inputs
    print(f'{n_inputs=}', f'{n_delays=}')
    col_idx = jnp.arange(n_inputs).repeat(n_delays)
    delayed_x = delay_buffer[delays, col_idx]
    out = jnp.einsum('ij,ij->i', w, delayed_x)
    return (delay_buffer, out, delays, w), (out, delay_buffer)

def wrapper_delay_denram_w(n_in, n_out, d_max, mc_input):
    global d_dr, w_dr
    out_dr_w = jnp.zeros((n_out, ), dtype='float32')
    delay_buffer = jnp.zeros((d_max, n_in), dtype='int32')
    state = (delay_buffer, out_dr_w, d_dr, w_dr)
    _, (out_dr_w, _) = jax.lax.scan(delay_denram_w, state, mc_input)
    return out_dr_w

j_wrapper_delay_denram_w = jax.jit(wrapper_delay_denram_w, static_argnums=(0, 1, 2))

In [30]:
def wrapper_jvjvj_delay_mich_fft_kwrep(input, Kw):
    return jvjvj_delay_mich_fft_krep(input, Kw).sum(2).sum(1)

def wrapper_jvjv_delay_mich_fft_kwrep(input, Kw):
    return jvjv_delay_mich_fft_krep(input, Kw).sum(2).sum(1)

j_wrapper_jvjvj_delay_mich_fft_kwrep = jax.jit(wrapper_jvjvj_delay_mich_fft_kwrep)
j_wrapper_jvjv_delay_mich_fft_kwrep = jax.jit(wrapper_jvjv_delay_mich_fft_kwrep)

In [31]:
def wrapper_jvjvj_delay_mich_fft_kw(input, Kw):
    return jvjvj_delay_mich_fft_k(input, Kw).sum(1)

def wrapper_jvjv_delay_mich_fft_kw(input, Kw):
    return jvjv_delay_mich_fft_k(input, Kw).sum(1)

j_wrapper_jvjvj_delay_mich_fft_kw = jax.jit(wrapper_jvjvj_delay_mich_fft_kw)
j_wrapper_jvjv_delay_mich_fft_kw = jax.jit(wrapper_jvjv_delay_mich_fft_kw)

In [32]:
out_dr_w = j_wrapper_delay_denram_w(n_in, n_out, d_max, mc_input)
print(out_dr_w.shape, "= out_dr_w.shape")
out_kwrep_jvjvj = j_wrapper_jvjvj_delay_mich_fft_kwrep(mc_input_k, K_rep)
print(out_kwrep_jvjvj.shape, "= out_kwrep_jvjvj.shape")
out_kwrep_jvjv = j_wrapper_jvjv_delay_mich_fft_kwrep(mc_input_k, K_rep)
print(out_kwrep_jvjv.shape, "= out_kwrep_jvjv.shape")
out_kw_jvjvj = j_wrapper_jvjvj_delay_mich_fft_kw(mc_input_k, K)
print(out_kw_jvjvj.shape, "= out_kw_jvjvj.shape")
out_kw_jvjv = j_wrapper_jvjv_delay_mich_fft_kw(mc_input_k, K)
print(out_kw_jvjv.shape, "= out_kw_jvjv.shape")

n_inputs=700 n_delays=16
(150, 20) = out_dr_w.shape
input.shape=(150,)
K.shape=(16, 100)
(20, 150) = out_kwrep_jvjvj.shape
input.shape=(150,)
K.shape=(16, 100)
(20, 150) = out_kwrep_jvjv.shape
input.shape=(150,)
K.shape=(100,)
(20, 150) = out_kw_jvjvj.shape
input.shape=(150,)
K.shape=(100,)
(20, 150) = out_kw_jvjv.shape


In [37]:
absolute_tolerance = 1e-4
print(jnp.allclose(out_dr_w.T, out_kwrep_jvjvj, atol=absolute_tolerance))
print(jnp.allclose(out_dr_w.T, out_kw_jvjvj, atol=absolute_tolerance))
print(jnp.allclose(out_dr_w.T, out_kwrep_jvjv, atol=absolute_tolerance))
print(jnp.allclose(out_dr_w.T, out_kw_jvjv, atol=absolute_tolerance))

True
True
True
True


In [38]:
%timeit j_wrapper_delay_denram_w(n_in, n_out, d_max, mc_input)

1.19 ms ± 71.4 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [67]:
%timeit j_wrapper_jvjvj_delay_mich_fft_kwrep(mc_input_k, K_rep)

1.08 ms ± 157 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [68]:
%timeit j_wrapper_jvjv_delay_mich_fft_kwrep(mc_input_k, K_rep)

1.08 ms ± 113 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [69]:
%timeit j_wrapper_jvjvj_delay_mich_fft_kw(mc_input_k, K)

82.2 μs ± 13.3 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [70]:
%timeit j_wrapper_jvjv_delay_mich_fft_kw(mc_input_k, K)

82.2 μs ± 7.5 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [39]:
kw_time = 83.1
kwrep_time = 2980
denram_w_time = 1110
print(f'Kw allows to go {denram_w_time/kw_time:.2f} times faster')
print(f'Kw_rep is {kwrep_time/kw_time:.2f} times slower than Kw')

Kw allows to go 13.36 times faster
Kw_rep is 35.86 times slower than Kw


In [40]:
@jax.custom_jvp
def gr_than(x, thr):
    """ Thresholding function for spiking neurons. """
    return (x > thr).astype(jnp.float32)


@gr_than.defjvp
def gr_jvp(primals, tangents):
    """ Surrogate gradient function for thresholding. """
    x, thr = primals
    x_dot, y_dot = tangents
    primal_out = gr_than(x, thr)
    tangent_out = x_dot / (10 * jnp.absolute(x - thr) + 1)**2
    return primal_out, tangent_out

def delay_denram_w_LIF(state, x):
    w, delay_buffer, out_spikes, I_in, V_mem, delays = state[0]
    beta, max_delay, timestep, V_th, _, _ = state[1]
    
    delay_buffer = jnp.concatenate(
        [jax.lax.expand_dims(x, (0,)), delay_buffer[:-1,:]],
        0
    )
    n_inputs = delay_buffer.shape[1] 
    n_delays = delays.shape[-1] // n_inputs
    print(f'{n_inputs=}', f'{n_delays=}')
    col_idx = jnp.arange(n_inputs).repeat(n_delays)
    delayed_x = delay_buffer[delays, col_idx]
    I_in = jnp.einsum('ij,ij->i', w, delayed_x)
    V_mem = beta * V_mem + I_in
    V_mem = jnp.maximum(0, V_mem) # HW constraint
    out_spikes = gr_than(V_mem, V_th)
    updated_state = (w, delay_buffer, out_spikes, I_in, V_mem, delays)
    return (updated_state, state[1]), (I_in, V_mem, out_spikes)

def wrapper_delay_denram_w_LIF(n_in, n_out, d_max, mc_input):
    global d_dr, w_dr
    delay_buffer = jnp.zeros((d_max, n_in), dtype='int32')
    I_in = jnp.zeros((n_out, ), dtype='float32')
    V_mem = jnp.zeros((n_out, ), dtype='float32')
    out_spikes = jnp.zeros((n_out, ), dtype='float32')
    
    state = ((w_dr, delay_buffer, out_spikes, I_in, V_mem, d_dr), (jnp.exp(-0.005/0.04), d_max, 0.005, 1, 0, 0))
    _, (I_in_dr, V_mem_dr, out_spikes_dr) = jax.lax.scan(delay_denram_w_LIF, state, mc_input)
    return (I_in_dr, V_mem_dr, out_spikes_dr)

j_wrapper_delay_denram_w_LIF = jax.jit(wrapper_delay_denram_w_LIF, static_argnums=(0, 1, 2))

In [41]:
def LIF_model(state, input):
    w, out_spikes, V_mem = state[0]
    beta, timestep, V_th, _, _ = state[1]
    V_mem = beta * V_mem + input
    V_mem = jnp.maximum(0, V_mem) # HW constraint
    out_spikes = gr_than(V_mem, V_th)
    updated_state = (w, out_spikes, V_mem)
    return (updated_state, state[1]), (V_mem, out_spikes)

v_LIF_model = jax.vmap(LIF_model, in_axes=(0, 0))
jv_LIF_model = jax.jit(v_LIF_model)
    

def wrapper_jvjvj_delay_mich_fft_kwrep_LIF(input, Kwrep):
    I_in_kwrep = j_wrapper_jvjvj_delay_mich_fft_kwrep(input, Kwrep)
    V_mem_kwrep = jnp.zeros((n_out, ), dtype='float32')
    out_spikes_kwrep = jnp.zeros((n_out, ), dtype='float32')
    state = ((Kwrep, out_spikes_kwrep, V_mem_kwrep), (jnp.exp(-0.005/0.04), 0.005, 1, 0, 0))
    _, (V_mem_kwrep, out_spikes_kwrep) = jax.lax.scan(LIF_model, state, I_in_kwrep.T)
    return (I_in_kwrep, V_mem_kwrep, out_spikes_kwrep)

j_wrapper_jvjvj_delay_mich_fft_kwrep_LIF = jax.jit(wrapper_jvjvj_delay_mich_fft_kwrep_LIF)

#TODO: DISCLAIMER, exact same code as above, but with Kw instead of Kwrep
def wrapper_jvjvj_delay_mich_fft_kw_LIF(input, Kw):
    I_in_kw = j_wrapper_jvjvj_delay_mich_fft_kw(input, Kw) # THIS IS THE FUNCTION THAT DELAYS (THIS IS KEY!!!!)
    V_mem_kw = jnp.zeros((n_out, ), dtype='float32')
    out_spikes_kw = jnp.zeros((n_out, ), dtype='float32')
    state = ((Kw, out_spikes_kw, V_mem_kw), (jnp.exp(-0.005/0.04), 0.005, 1, 0, 0))
    _, (V_mem_kw, out_spikes_kw) = jax.lax.scan(LIF_model, state, I_in_kw.T)
    return (I_in_kw, V_mem_kw, out_spikes_kw)

j_wrapper_jvjvj_delay_mich_fft_kw_LIF = jax.jit(wrapper_jvjvj_delay_mich_fft_kw_LIF)

In [42]:
n_in = 700
n_out = 20
n_rep = 16
sim_len = 150
d_max = 30


# mc_input = jnp.arange(1, sim_len+1)[:, None] + jnp.arange(n_in)[None, :] * sim_len
# print(mc_input.T, "= multi_channel_input.T")
# mc_input_k = mc_input.T
key = jax.random.PRNGKey(0)
mc_input = jax.random.randint(key, minval=0, maxval=2, shape=(sim_len, n_in))
mc_input_k = mc_input.T
print(mc_input_k.shape, "= mc_input_k.shape")
print(mc_input.min(), mc_input.max(), "= min, max")

dk = delay_k_generation(n_in, n_rep, n_out, d_max)
print(dk.shape, "= dk.shape")
d_dr = delay_from_kernel_to_denram(dk)
print(d_dr.shape, "= d_dr.shape")

w = weight_generation(key, n_out, n_in, n_rep)
# w_dr = weight_from_kernel_to_denram(w)
print(w.shape, "= w.shape")
# print(w_dr.shape, "= w_dr.shape")

K, K_rep = kw_generation(n_in, n_rep, n_out, d_max, dk, w)
print(K.shape) #(n_out, n_in, d_max)

(700, 150) = mc_input_k.shape
0 1 = min, max
(700, 16)
[[ 0  1  2 ... 13 14 15]
 [ 1  2  3 ... 14 15 16]
 [ 2  3  4 ... 15 16 17]
 ...
 [ 7  8  9 ... 20 21 22]
 [ 8  9 10 ... 21 22 23]
 [ 9 10 11 ... 22 23 24]]
(20, 700, 16) = dk.shape
(20, 11200) = d_dr.shape
(20, 700, 16) = w.shape
True
(20, 700, 30)


In [43]:
(I_in_dr, V_mem_dr, out_spikes_dr) = j_wrapper_delay_denram_w_LIF(n_in, n_out, d_max, mc_input)
print(I_in_dr.shape, V_mem_dr.shape, out_spikes_dr.shape)

n_inputs=700 n_delays=16
(150, 20) (150, 20) (150, 20)


In [44]:
print(I_in_dr[0,:10])

[  0.58   4.17  -1.02  -4.64 -12.36   5.3   -5.12   6.34   9.14  -0.86]


In [45]:
(I_in_kwrep, V_mem_kwrep, out_spikes_kwrep) = j_wrapper_jvjvj_delay_mich_fft_kwrep_LIF(mc_input_k, K_rep)
print(I_in_kwrep.shape, V_mem_kwrep.shape, out_spikes_kwrep.shape)

input.shape=(150,)
K.shape=(16, 30)
(20, 150) (150, 20) (150, 20)


In [46]:
(I_in_kw, V_mem_kw, out_spikes_kw) = j_wrapper_jvjvj_delay_mich_fft_kw_LIF(mc_input_k, K)
print(I_in_kw.shape, V_mem_kw.shape, out_spikes_kw.shape)

input.shape=(150,)
K.shape=(30,)
(20, 150) (150, 20) (150, 20)


In [47]:
absolute_tolerance = 1e-4
print(jnp.allclose(I_in_dr.T, I_in_kwrep, atol=absolute_tolerance))
print(jnp.allclose(I_in_dr.T, I_in_kw, atol=absolute_tolerance))
print(jnp.allclose(I_in_kwrep, I_in_kw, atol=absolute_tolerance))

True
True
True


In [48]:
%timeit j_wrapper_delay_denram_w_LIF(n_in, n_out, d_max, mc_input)

1.5 ms ± 66.2 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [ ]:
%timeit j_wrapper_jvjvj_delay_mich_fft_kwrep_LIF(mc_input_k, K_rep)

In [ ]:
%timeit j_wrapper_jvjvj_delay_mich_fft_kw_LIF(mc_input_k, K)

In [ ]:
denram_w_LIF_time = 1420
kwrep_LIF_time = 3780
kw_LIF_time = 992
print(f'Kw allows to go {denram_w_LIF_time/kw_LIF_time:.2f} times faster')
print(f'Kw_rep is {kwrep_LIF_time/kw_LIF_time:.2f} times slower than Kw')

In [ ]:
d_max_list = [30, 100, 300]
denram_w_LIF_time = [1390, 1420, 1550]
kwrep_LIF_time = [2970, 3780, 6110]
kw_LIF_time = [966, 992, 980]
speed_up = [denram_w_LIF_time[i]/kw_LIF_time[i] for i in range(3)]

In [ ]:
import matplotlib.pyplot as plt
px = 1/plt.rcParams['figure.dpi']

fig, ax = plt.subplots(1, 1, figsize=(400*px, 200*px))

ax.plot(d_max_list, denram_w_LIF_time, label='DenRAM')
# ax.plot(d_max_list, kwrep_LIF_time, label='Kw_rep')
ax.plot(d_max_list, kw_LIF_time, label='K')

ax.set_xlabel('d_max')
ax.set_ylabel('Time (us)')
ax.set_title('Time comparison')
ax.legend()

# set y-axis to 200 us ticks
ax.set_yticks(np.arange(800, 1600, 200))
# set x-axis range to 0-350
ax.set_xlim(0, 350)

# add above the lines the text with the speed up
for i in range(3):
    ax.text(d_max_list[i], kw_LIF_time[i], f'{speed_up[i]:.2f}', ha='center', va='top')

ax.grid()
plt.show()

In [ ]:
n_in = 700
n_out = 20
n_rep = 16
sim_len = 150
d_max = 30

bs = 64


# mc_input = jnp.arange(1, sim_len+1)[:, None] + jnp.arange(n_in)[None, :] * sim_len
# print(mc_input.T, "= multi_channel_input.T")
# mc_input_k = mc_input.T
key = jax.random.PRNGKey(0)
mc_input = jax.random.randint(key, minval=0, maxval=2, shape=(bs, sim_len, n_in))
mc_input_k = jnp.swapaxes(mc_input, 1, 2)
print(mc_input_k.shape, "= mc_input_k.shape")
print(mc_input.min(), mc_input.max(), "= min, max")

dk = delay_k_generation(n_in, n_rep, n_out, d_max)
print(dk.shape, "= dk.shape")
d_dr = delay_from_kernel_to_denram(dk)
print(d_dr.shape, "= d_dr.shape")

w = weight_generation(key, n_out, n_in, n_rep)
w_dr = weight_from_kernel_to_denram(w)
print(w.shape, "= w.shape")
print(w_dr.shape, "= w_dr.shape")

K, K_rep = kw_generation(n_in, n_rep, n_out, d_max, dk, w)

In [ ]:
j_wrapper_delay_denram_w_LIF = jax.jit(wrapper_delay_denram_w_LIF, static_argnums=(0, 1, 2))
v_wrapper_delay_denram_w_LIF = jax.vmap(j_wrapper_delay_denram_w_LIF, in_axes=(None, None, None, 0))
j_wrapper_jvjvj_delay_mich_fft_kwrep_LIF = jax.jit(wrapper_jvjvj_delay_mich_fft_kwrep_LIF)
v_wrapper_jvjvj_delay_mich_fft_kwrep_LIF = jax.vmap(j_wrapper_jvjvj_delay_mich_fft_kwrep_LIF, in_axes=(0, None))
j_wrapper_jvjvj_delay_mich_fft_kw_LIF = jax.jit(wrapper_jvjvj_delay_mich_fft_kw_LIF)
v_wrapper_jvjvj_delay_mich_fft_kw_LIF = jax.vmap(j_wrapper_jvjvj_delay_mich_fft_kw_LIF, in_axes=(0, None))

vj_wrapper_delay_denram_w_LIF = jax.vmap(j_wrapper_delay_denram_w_LIF, in_axes=(None, None, None, 0))
vj_wrapper_jvjvj_delay_mich_fft_kwrep_LIF = jax.vmap(j_wrapper_jvjvj_delay_mich_fft_kwrep_LIF, in_axes=(0, None))
vj_wrapper_jvjvj_delay_mich_fft_kw_LIF = jax.vmap(j_wrapper_jvjvj_delay_mich_fft_kw_LIF, in_axes=(0, None))

jv_wrapper_delay_denram_w_LIF = jax.jit(v_wrapper_delay_denram_w_LIF, static_argnums=(0, 1, 2))
jv_wrapper_jvjvj_delay_mich_fft_kwrep_LIF = jax.jit(v_wrapper_jvjvj_delay_mich_fft_kwrep_LIF)
jv_wrapper_jvjvj_delay_mich_fft_kw_LIF = jax.jit(v_wrapper_jvjvj_delay_mich_fft_kw_LIF)

In [ ]:
(I_in_dr_jv, V_mem_dr_jv, out_spikes_dr_jv) = jv_wrapper_delay_denram_w_LIF(n_in, n_out, d_max, mc_input)
print(I_in_dr_jv.shape, V_mem_dr_jv.shape, out_spikes_dr_jv.shape)
(I_in_kwrep_jv, V_mem_kwrep_jv, out_spikes_kwrep_jv) = jv_wrapper_jvjvj_delay_mich_fft_kwrep_LIF(mc_input_k, K_rep)
print(I_in_kwrep_jv.shape, V_mem_kwrep_jv.shape, out_spikes_kwrep_jv.shape)
(I_in_kw_jv, V_mem_kw_jv, out_spikes_kw_jv) = jv_wrapper_jvjvj_delay_mich_fft_kw_LIF(mc_input_k, K)
print(I_in_kw_jv.shape, V_mem_kw_jv.shape, out_spikes_kw_jv.shape)

In [191]:
def delay_mich_fft_k(input, K):
    '''
    :param input: 1D array of shape (sim_len,)
    :param K: 1D array of shape (d_max,). n_rep-hot encoded delays
    :return: 1D array of shape (sim_len+d_max,)
    '''
    # print(f'{input.shape=}')
    # print(f'{K.shape=}')
    d_max = K.shape[-1]
    sim_len = input.shape[-1]
    print("sim_len", sim_len)
    input_fft = jnp.fft.rfft(jnp.pad(input, (0, d_max)))
    print(input.shape, " input shape")
    print(input_fft.shape, " input_fft shape")
    K_fft = jnp.fft.rfft(jnp.pad(K, (0, sim_len)))
    print(K.shape, " K shape")
    print(K_fft.shape, " K_fft shape")
    I_fft = K_fft * input_fft
    I = jnp.fft.irfft(I_fft)[:input.shape[0]]
    return I

v_delay_mich_fft_k = jax.vmap(delay_mich_fft_k, in_axes=(0, 0))
v_v_delay_mich_fft_k = jax.vmap(v_delay_mich_fft_k, in_axes=(None, 0))


j_delay_mich_fft_k = jax.jit(delay_mich_fft_k)
vj_delay_mich_fft_k = jax.vmap(j_delay_mich_fft_k, in_axes=(0, 0))
jvj_delay_mich_fft_k = jax.jit(vj_delay_mich_fft_k)
vjvj_delay_mich_fft_k = jax.vmap(jvj_delay_mich_fft_k, in_axes=(None, 0))
jvjvj_delay_mich_fft_k = jax.jit(vjvj_delay_mich_fft_k)

In [192]:
def wrapper_jvjvj_delay_mich_fft_kw(input, Kw):
    return jvjvj_delay_mich_fft_k(input, Kw).sum(1)

def wrapper_jvjv_delay_mich_fft_kw(input, Kw):
    return jvjv_delay_mich_fft_k(input, Kw).sum(1)

j_wrapper_jvjvj_delay_mich_fft_kw = jax.jit(wrapper_jvjvj_delay_mich_fft_kw)
j_wrapper_jvjv_delay_mich_fft_kw = jax.jit(wrapper_jvjv_delay_mich_fft_kw)

In [193]:
def uniform_delay_k_generation(key, n_out, n_in, n_rep, d_max):
    key, subkey = jax.random.split(key)
    discrete_delays = (
        jax.random.randint(subkey, (n_out, n_in, n_rep),
                           minval=0,
                           maxval=d_max + 1))
    return key, discrete_delays

In [194]:
def k_generation(n_in, n_rep, n_out, d_max, delays):
    out_idx = jnp.tile(jnp.ones((n_in, n_rep), dtype='int32'), (n_out, 1, 1)) * jnp.arange(n_out)[:, None, None]
    in_idx = jnp.ones((n_in, n_rep), dtype='int32') * jnp.arange(n_in)[:, None]
    rep_idx = jnp.tile(jnp.arange(n_rep), (n_in, 1))
    K_rep = jnp.zeros((n_out, n_in, n_rep, d_max), dtype='int32')
    K_rep = K_rep.at[out_idx, in_idx, rep_idx, delays].set(1)
    K = jnp.zeros((n_out, n_in, d_max), dtype='int32')
    K = K.at[out_idx, in_idx, delays].set(1)
    print(jnp.array_equal(K_rep.sum(2), K))
    return K, K_rep

In [195]:
def weight_generation(key, n_out, n_in, n_rep):
    w = jax.random.normal(key, (n_out, n_in, n_rep))
    return w

In [196]:
def kw_generation(n_in, n_rep, n_out, d_max, delays, weights):
    out_idx = jnp.tile(jnp.ones((n_in, n_rep), dtype='int32'), (n_out, 1, 1)) * jnp.arange(n_out)[:, None, None]
    in_idx = jnp.ones((n_in, n_rep), dtype='int32') * jnp.arange(n_in)[:, None]
    rep_idx = jnp.tile(jnp.arange(n_rep), (n_in, 1))
    K_rep = jnp.zeros((n_out, n_in, n_rep, d_max), dtype='float32')
    K_rep = K_rep.at[out_idx, in_idx, rep_idx, delays].set(weights)
    K = jnp.zeros((n_out, n_in, d_max), dtype='float32')
    K = K.at[out_idx, in_idx, delays].set(weights)
    print(jnp.array_equal(K_rep.sum(2), K))
    return K, K_rep

In [197]:
n_in = 3
n_out = 2
n_rep = 4
sim_len = 20
d_max = 5



mc_input = jnp.arange(1, sim_len+1)[:, None] + jnp.arange(n_in)[None, :] * sim_len
mc_input = jnp.zeros((sim_len, n_in), dtype='int32')
mc_input=mc_input.at[1,0].set(1.0)
mc_input=mc_input.at[1,1].set(1.0)
mc_input=mc_input.at[1,2].set(1.0)
print(mc_input.T.shape, " = multi_channel_input.T.shape")
print(mc_input.T, "= multi_channel_input.T")
mc_input_k = mc_input.T

dk = delay_k_generation(n_in, n_rep, n_out, d_max)
print(dk.shape, "= dk.shape") #(n_out, n_in, n_rep)

w = jnp.ones((n_out, n_in, n_rep))#weight_generation(key, n_out, n_in, n_rep) #(n_out, n_in, n_rep)
print(w.shape, "= w.shape") #(n_out, n_in, n_rep)

Kw, _ = kw_generation(n_in, n_rep, n_out, d_max, dk, w) #(n_ut, n_oin, d_max)
print(Kw.shape, "= Kw.shape")

# K, _ = k_generation(n_in, n_rep, n_out, d_max, dk)
# print(K.shape, "= K.shape") #(n_out, n_in, d_max)

(3, 20)  = multi_channel_input.T.shape
[[0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]] = multi_channel_input.T
(3, 4)
[[0 1 2 3]
 [1 2 3 4]
 [0 1 2 3]]
(2, 3, 4) = dk.shape
(2, 3, 4) = w.shape
True
(2, 3, 5) = Kw.shape


In [198]:
# actually delaying the input
# input.shape = (n_in, sim_len)


I_in_kw = delay_mich_fft_k(mc_input_k, Kw).sum(1)


print(mc_input_k.shape, "= mc_input_k.shape")
print(I_in_kw.shape, "= I_in_kw.shape")
print(mc_input_k)
print(I_in_kw)

sim_len 20
(3, 20)  input shape
(8, 13)  input_fft shape
(2, 3, 5)  K shape
(22, 23, 13)  K_fft shape


ValueError: Incompatible shapes for broadcasting: shapes=[(22, 23, 13), (8, 13)]